In [6]:
from google.colab import drive


In [19]:
import os

print(os.listdir("/content/drive/MyDrive/Colab Notebooks"))

['Day3_Agentic_AI.ipynb', '.ipynb_checkpoints', 'Untitled2.ipynb', 'Untitled0.ipynb', 'Untitled1.ipynb', 'nlp_tokenization.ipynb', 'Customer_Segmentation_using_KMeans.ipynb', 'Untitled3.ipynb', 'Untitled4.ipynb', 'Copy of Day 1 Agentic AI.ipynb', 'Day 1 Agentic AI.ipynb', 'day1_code.py', 'Day2_Agentic_AI.ipynb', 'day2_code.py']


In [20]:
%run "/content/drive/MyDrive/Colab Notebooks/day2_code.py"

In [21]:
!pip install -q langchain langchain-groq

In [22]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.3
)

In [24]:
SYSTEM = """ROLE:
You are VulnAgent, an AI-based vulnerability assessment
and prioritization assistant.

FACTS:
The user provides vulnerability descriptions.
Analyze only the information provided by the user.
Consider vulnerability type, possible impact, severity,
and remediation.

RULES:
Do not claim that you scanned or tested a real system.
Do not invent scan results or CVSS scores.
If information is insufficient, clearly mention it.
Provide defensive security recommendations.
Do not provide instructions to exploit real systems.

FORMAT:
Return a clear vulnerability analysis containing:
Vulnerability Name
Severity
Impact
Remediation
Verification
Confidence

Keep the response simple, structured, and understandable."""

In [26]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

In [28]:
history = []

def ask(question: str) -> str:
    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer

In [30]:
print(ask("""
A web application login page is vulnerable to SQL injection.
User input is directly inserted into SQL queries.
Analyze the vulnerability and suggest remediation.
"""))

**Vulnerability Name**  
SQL Injection in Login Page  

**Severity**  
High – allows attackers to manipulate authentication queries and potentially gain unauthorized access.

**Impact**  
- Bypass of user authentication (login bypass).  
- Unauthorized read, modify, or delete of database records.  
- Possible escalation to full database compromise if privileged DB accounts are used.  
- Data leakage, loss of integrity, and potential compliance violations.

**Remediation**  
1. **Use Parameterized Queries / Prepared Statements**  
   - Replace string concatenation with bound parameters (e.g., `PreparedStatement` in Java, `PDO` prepared statements in PHP, `SqlCommand` with parameters in .NET).  
2. **Adopt an ORM or Query Builder** that abstracts raw SQL and handles escaping automatically.  
3. **Input Validation**  
   - Apply strict allow‑list validation (e.g., only alphanumeric characters for usernames).  
   - Reject or properly encode any characters that could affect SQL syntax.  
4

In [32]:
print(ask("What vulnerability did I ask you to analyze?"))

You asked me to analyze a **SQL Injection** vulnerability.


In [39]:
import csv

test_data = [
    {
        "question": "What is SQL Injection?",
        "must_include": "sql injection,database"
    },
    {
        "question": "What is a weak password?",
        "must_include": "weak password,brute force"
    },
    {
        "question": "What are security headers?",
        "must_include": "security headers,browser"
    },
    {
        "question": "Explain Remote Code Execution.",
        "must_include": "remote code execution,attacker"
    },
    {
        "question": "How can SQL injection be prevented?",
        "must_include": "parameterized queries,input validation"
    }
]

with open("vulnagent_tests.csv", "w", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["question", "must_include"]
    )

    writer.writeheader()
    writer.writerows(test_data)

print("Test dataset created successfully!")

Test dataset created successfully!


In [40]:
import csv

def run_tests(path="vulnagent_tests.csv"):
    passed = 0
    total = 0

    with open(path, "r", encoding="utf-8") as file:
        reader = csv.DictReader(file)

        for row in reader:
            question = row["question"]

            expected_keywords = [
                keyword.strip().lower()
                for keyword in row["must_include"].split(",")
            ]

            try:
                response = chain.invoke({
                    "history": [],
                    "question": question
                })

                response = response.lower()

                missing_keywords = [
                    keyword
                    for keyword in expected_keywords
                    if keyword not in response
                ]

                total += 1

                if len(missing_keywords) == 0:
                    passed += 1
                    print("PASS:", question)
                else:
                    print("FAIL:", question)
                    print("Missing:", missing_keywords)

            except Exception as error:
                total += 1
                print("ERROR:", question)
                print(error)

    print("\nVulnAgent Test Results")
    print("---------------------")
    print("Passed:", passed)
    print("Failed:", total - passed)
    print("Total:", total)

    if total > 0:
        pass_rate = (passed / total) * 100
        print("Pass Rate:", round(pass_rate, 2), "%")

run_tests()

PASS: What is SQL Injection?
FAIL: What is a weak password?
Missing: ['brute force']
PASS: What are security headers?
PASS: Explain Remote Code Execution.
PASS: How can SQL injection be prevented?

VulnAgent Test Results
---------------------
Passed: 4
Failed: 1
Total: 5
Pass Rate: 80.0 %


In [38]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Explain SQL Injection and its prevention."
        }
    ]
})

print(result["messages"][-1].content)

**Vulnerability Name:** SQL Injection (SQLi)  

**Severity:** High (potentially Critical) – depends on the application’s exposure, data sensitivity, and privilege level of the compromised database account.  

**Impact**  
- Unauthorized read, modify, or delete of database records.  
- Disclosure of sensitive data (e.g., PII, credentials, financial information).  
- Bypass of authentication or authorization controls.  
- Execution of administrative commands (e.g., DROP TABLE, CREATE USER).  
- Potential pivot to other systems if the database server is on a trusted network.  

**Remediation**  
1. **Use Parameterized Queries / Prepared Statements**  
   - Bind user‑supplied values to placeholders rather than concatenating them into SQL strings.  
   - Supported by most languages/frameworks (e.g., `PreparedStatement` in Java, `PDO` in PHP, `SqlCommand` with parameters in .NET).  

2. **Employ Stored Procedures** (when they accept parameters safely) – avoid building dynamic SQL inside the 

In [41]:
print(ask("""
A web application allows users to create passwords such as
123456 and password123.
There is no password strength validation.

Analyze the vulnerability and suggest remediation.
"""))

**Vulnerability Name**  
Weak Password Policy (No Strength Validation)  

**Severity**  
Medium‑High – while the flaw itself does not give immediate remote code execution, it greatly increases the risk of account compromise through credential‑guessing, credential‑stuffing, and brute‑force attacks.

**Impact**  
- **Account takeover**: Simple passwords (e.g., “123456”, “password123”) can be guessed or cracked quickly, giving attackers unauthorized access to user accounts.  
- **Privilege escalation**: If administrative or privileged accounts can use weak passwords, the attacker may gain higher‑level access to the application and underlying systems.  
- **Data breach**: Compromised accounts can lead to exposure or manipulation of sensitive data stored in the application.  
- **Reputation & compliance**: Many regulations (e.g., GDPR, PCI‑DSS, NIST) require reasonable password security; failure to enforce it may result in non‑compliance findings.

**Remediation**  
1. **Enforce Password Co